#  Projekt 7 - Optymalizacja modeli Ultralytics YOLO

Poniższy skrypt należy uruchomić w środowisku Google Colab.

## Model

*YOLO26n* jest najmniejszym modelem z rodziny YOLO26 (*You-Only-Look-Once*). Jest to model real-time detekcji obiektów. 

In [1]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")

## Pomiar czasu wnioskowania

Wykorzystując poniższy skrypt możemy w prosty sposób zmierzyć czas wnioskowania modelu.

In [2]:
import time
import cv2 as cv, numpy as np

def inference(model: YOLO, image: np.ndarray): model(image, verbose=False, device="cpu", imgsz=640)

def measure_inference_time(model: YOLO, image_path: str, n_iterations: int=100):

    # Load image
    image = cv.imread(image_path)

    # Coldstart - pierwsze wnioskowanie, zazwyczaj wolniejsze ze względu na inicjalizację modelu i wczytanie wag
    start = time.time()
    inference(model,image)
    coldstart_time = time.time() - start
    
    # Pomiar dla N iteracji
    times = []
    for _ in range(n_iterations):
        start = time.time()
        inference(model,image)
        times.append(time.time() - start)
    
    avg_time = sum(times) / len(times)
    
    return {
        "average_inference_time": avg_time * 1000,
        "min_time": min(times) * 1000,
        "max_time": max(times) * 1000,
    }

In [3]:
timing_results = measure_inference_time(model, "bus.jpg", n_iterations=10)
print(f"Average inference time: {timing_results['average_inference_time']:.1f}ms")
print(f"Min time: {timing_results['min_time']:.1f}ms")
print(f"Max time: {timing_results['max_time']:.1f}ms")


Average inference time: 107.2ms
Min time: 101.6ms
Max time: 117.5ms


Czy istnieją sposoby na zmniejszenie czasu wnioskowania (poprawę wydajności)?

### Zapis do formatu ONNX

Format *ONNX* działa jak uniwersalny tłumacz w świecie sztucznej inteligencji, który pozwala uniezależnić modele od konkretnego środowiska programistycznego. Zamiast zmuszać system docelowy do instalowania ciężkich bibliotek takich jak PyTorch czy TensorFlow, model jest eksportowany do otwartego standardu opartego na uniwersalnym grafie obliczeniowym. Dzięki temu ten sam plik możesz bezproblemowo uruchomić na procesorach Intel, AMD, architekturze ARM (np. Raspberry Pi) czy układach graficznych NVIDIA, używając do tego lekkiego i zoptymalizowanego silnika ONNX Runtime.

### Kwantyzacja

Kwantyzacja *INT8* to z kolei proces drastycznej optymalizacji matematycznej, który polega na zamianie 32-bitowych wag zmiennoprzecinkowych (FP32) na 8-bitowe liczby całkowite (INT8). W praktyce oznacza to "zaokrąglenie" skomplikowanych ułamków do prostych wartości z zakresu od -128 do 127, co natychmiast zmniejsza rozmiar pliku modelu aż o 75% i drastycznie redukuje zużycie pamięci RAM. Ponieważ sprzęt komputerowy przetwarza operacje na liczbach całkowitych o wiele szybciej i przy mniejszym poborze energii, model zyskuje potężny przyrost klatek na sekundę (FPS) przy jednoczesnym zachowaniu niemal identycznej dokładności detekcji.

In [4]:
model.export(
    format="onnx",
    data="coco8.yaml",
    quantize=8,
) # ONNX + INT8

Ultralytics 8.4.98  Python-3.13.5 torch-2.7.1+cu128 CPU (Intel Xeon Silver 4110 2.10GHz)

PyTorch: starting from 'yolo26n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (5.3 MB)

ONNX: starting export with onnx 1.22.0 opset 19...


c:\Users\Kacper-Marciniak\anaconda3\envs\env_ml_new\Lib\site-packages\torch\onnx\symbolic_opset9.py:5350: UserWarning: Exporting aten::index operator of advanced indexing in opset 19 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  warnings.warn(


ONNX: slimming with onnxslim 0.1.94...
ONNX: collecting INT8 calibration images from 'data=coco8.yaml'
val: Fast image access  (ping: 0.10.0 ms, read: 243.0135.1 MB/s, size: 54.0 KB)
val: Scanning Z:\GitRepo\TCM-Hob-Inspection\Datasets\coco8\labels\val.cache... 4 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4/4 541.2Kit/s 0.0s
WARNING ONNX: >300 images recommended for INT8 calibration, found 4 images.
ONNX: quantizing INT8 with ONNX Runtime...


ONNX: export success  10.0s, saved as 'yolo26n_int8.onnx' (2.8 MB)

Export complete (10.4s)
Results saved to P:\Dydaktyka\Edge AI\P7\yolo26n_int8.onnx
Predict:         yolo predict task=detect model=yolo26n_int8.onnx imgsz=640 
Validate:        yolo val task=detect model=yolo26n_int8.onnx imgsz=640 data=/home/lq/codes/ultralytics/ultralytics/cfg/datasets/coco.yaml  
Visualize:       https://netron.app


'yolo26n_int8.onnx'

In [5]:
model_onnx_int8 = YOLO("yolo26n_int8.onnx", task="detect")

timing_results = measure_inference_time(model_onnx_int8, "bus.jpg", n_iterations=10)
print(f"Average inference time: {timing_results['average_inference_time']:.1f}ms")
print(f"Min time: {timing_results['min_time']:.1f}ms")
print(f"Max time: {timing_results['max_time']:.1f}ms")

Loading yolo26n_int8.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.27.0 with CPUExecutionProvider
Average inference time: 160.8ms
Min time: 149.6ms
Max time: 169.4ms


Na nowoczesnym PC możemy zaobserwować pogorszenie wydajności. Jaki będzie zysk czasowy na mikro-komputerze Raspberry Pi?